In [0]:
CREATE OR REPLACE VIEW workspace.default.rat_clean_v2 AS
WITH parsed AS (
  SELECT
    CAST(unique_key AS STRING)                  AS complaint_id,
    CAST(created_date AS TIMESTAMP)             AS created_date,
    CAST(closed_date AS TIMESTAMP)              AS closed_date,
    status,
    complaint_type,
    descriptor,
    location_type,
    LPAD(CAST(incident_zip AS STRING), 5, '0')  AS zip_code,
    INITCAP(TRIM(borough))                      AS borough,
    CAST(latitude AS DOUBLE)                    AS latitude,
    CAST(longitude AS DOUBLE)                   AS longitude
  FROM workspace.default.rat_sightings
  WHERE UPPER(TRIM(borough)) IN (
    'MANHATTAN',
    'BROOKLYN',
    'QUEENS',
    'BRONX',
    'STATEN ISLAND'
  )
    AND incident_zip BETWEEN 10001 AND 11697
    AND CAST(created_date AS TIMESTAMP) IS NOT NULL
)

SELECT
  *,

  CASE
    WHEN descriptor = 'Signs of Rodents'
      THEN 'Inspector-recorded signs'
    WHEN descriptor IN (
      'Rat Sighting',
      'Condition Attracting Rodents',
      'Mouse Sighting'
    )
      THEN 'Resident-reported 311'
    ELSE 'Other or unknown'
  END AS provenance_type,

  descriptor IN (
    'Rat Sighting',
    'Condition Attracting Rodents',
    'Mouse Sighting'
  ) AS resident_report,

  descriptor IN (
    'Rat Sighting',
    'Condition Attracting Rodents'
  ) AS resident_rat_report,

  descriptor = 'Rat Sighting'
    AS direct_rat_sighting,

  descriptor = 'Condition Attracting Rodents'
    AS attracting_condition_report,

  descriptor = 'Mouse Sighting'
    AS resident_mouse_report,

  descriptor = 'Signs of Rodents'
    AS inspector_signs_record,

  DATE_TRUNC('MONTH', created_date)
    AS created_month,

  CASE
    WHEN closed_date IS NOT NULL
    THEN UNIX_TIMESTAMP(closed_date) - UNIX_TIMESTAMP(created_date)
    ELSE NULL
  END AS closure_seconds,

  CASE
    WHEN closed_date IS NOT NULL
      AND UNIX_TIMESTAMP(closed_date) - UNIX_TIMESTAMP(created_date)
          BETWEEN 0 AND 60
    THEN TRUE
    ELSE FALSE
  END AS closed_within_60_seconds,

  CASE
    WHEN latitude IS NOT NULL AND longitude IS NOT NULL
    THEN CONCAT(
      CAST(ROUND(latitude, 6) AS STRING),
      ',',
      CAST(ROUND(longitude, 6) AS STRING)
    )
    ELSE NULL
  END AS coordinate_key

FROM parsed;

SELECT
  provenance_type,
  descriptor,
  COUNT(*) AS record_count
FROM workspace.default.rat_clean_v2
GROUP BY provenance_type, descriptor
ORDER BY provenance_type, record_count DESC;

WITH descriptor_counts AS (
  SELECT
    descriptor,
    COUNT(*) AS record_count
  FROM workspace.default.rat_clean_v2
  WHERE zip_code = '10035'
  GROUP BY descriptor
)

SELECT
  descriptor,
  record_count,
  ROUND(
    record_count * 100.0 / SUM(record_count) OVER (),
    2
  ) AS percentage_of_zip_records
FROM descriptor_counts
ORDER BY record_count DESC;

SELECT
  COUNT(*) AS total_clean_records,
  COUNT_IF(resident_report) AS all_resident_reports,
  COUNT_IF(resident_rat_report) AS resident_rat_reports,
  COUNT_IF(direct_rat_sighting) AS direct_rat_sightings,
  COUNT_IF(attracting_condition_report) AS attracting_condition_reports,
  COUNT_IF(resident_mouse_report) AS mouse_reports,
  COUNT_IF(inspector_signs_record) AS inspector_signs_records
FROM workspace.default.rat_clean_v2;

SELECT
  created_month,
  COUNT(*) AS closed_records,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY closure_seconds) AS median_closure_seconds,
  ROUND(
    100.0 * COUNT_IF(closed_within_60_seconds) / COUNT(*),
    2
  ) AS percent_closed_within_60s
FROM workspace.default.rat_clean_v2
WHERE resident_rat_report
  AND closure_seconds IS NOT NULL
GROUP BY created_month
ORDER BY created_month;